# Module 1 · Lesson 02: Your First OpenAI API Call

Welcome! In this notebook you will learn how to **communicate with OpenAI's GPT models** through their API.

## What you will learn
1. How to initialize the OpenAI client
2. The anatomy of a **chat completion** request
3. Using **system prompts** to control behaviour
4. How **temperature** affects creativity
5. Building **multi-turn conversations**

---

### Prerequisites
Make sure you have run `01_setup_verification.py` and that your `OPENAI_API_KEY` is set in `.env`.

OpenAI API Key: https://platform.openai.com/api-keys

OpenAI API Pricing: https://openai.com/api/pricing/

In [1]:
import os

from pathlib import Path
from dotenv import  load_dotenv
from IPython.display import display, Markdown

load_dotenv(Path.cwd().parent / "API_Verifications/.env")

from openai import OpenAI

client = OpenAI()

if client:
    display(Markdown("Client Ready"))

    


Client Ready

---
## 1. Basic Completion — Your Very First Call

The `chat.completions.create()` method is the **core building block** of every LLM application.

Three required parameters:
| Parameter | Purpose |
|-----------|--------|
| `model`   | Which GPT model to use |
| `messages` | The conversation so far (list of dicts) |
| `max_tokens` | Maximum length of the response |

Each message has a `role` (`"system"`, `"user"`, or `"assistant"`) and `content`.

In [ ]:
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {"role": "user", "content": "What is Python? Answer in one sentence."}
    ],
    max_tokens= 200
)

# Extract answer
#print(response)
answer = response.choices[0].message.content
#print(answer)
display(Markdown(f"**Response:** {answer}"))

# Token usage
u = response.usage
print(f"tokens - Prompt: {u.prompt_tokens}, Copletion: {u.completion_tokens}, Total: {u.total_tokens}")

ChatCompletion(id='chatcmpl-DJGgYr0CPttJwoL6Jt0TApwLV2UL4', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Python is a high-level, interpreted programming language known for its simplicity and readability, making it popular for various applications, including web development, data analysis, artificial intelligence, and automation.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1773484742, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_7cd1a06d3a', usage=CompletionUsage(completion_tokens=36, prompt_tokens=16, total_tokens=52, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
Python is a high-level, interpreted programming language known for its

> **Key Insight:** The API is *stateless* — it does not remember previous calls.
> Every request must contain the full conversation context.

---
## 2. System Prompts — Controlling Behaviour

A **system prompt** sets the AI's persona, tone, and constraints *before* the user's message.
Think of it as the "instruction manual" the model follows.

In [3]:
response = client.chat.completions.create(
    model= "gpt-4o-mini",
    messages= [
        {
            "role": "system",
            "content": (
                "You are a helpful programming tutor."
                "Explain consepts simply using analogies."
                "Keep responces concise (max 3 sentences)" 
            )
        },
        {
            "role": "user",
            "content": "What is recursion?"
        }
    ],
    max_tokens= 500
)

display(Markdown(f"### Tutor Response\n\n {response.choices[0].message.content}"))

# Token usage
u = response.usage
print(f"tokens - Prompt: {u.prompt_tokens}, Copletion: {u.completion_tokens}, Total: {u.total_tokens}")

### Tutor Response

 Recursion is like a set of Russian nesting dolls, where each doll contains a smaller one inside. In programming, a recursive function calls itself to solve smaller instances of the same problem until reaching a base case, much like opening each doll until you find the smallest one. This process continues until the entire problem is solved, just as the dolls are revealed one by one.

tokens - Prompt: 40, Copletion: 74, Total: 114


> **Best Practice:** Always include a system prompt in production.
> It improves consistency, safety, and output quality.

---
## 3. Temperature — Creativity vs Determinism

| Temperature | Behaviour | Use Case |
|-------------|-----------|----------|
| `0.0` | Deterministic, repeatable | Classification, extraction, math |
| `0.3–0.7` | Balanced | General Q&A, summarisation |
| `1.0` | Creative, varied | Brainstorming, storytelling |

Let's compare:

In [4]:
prompt = "Write a short story(around two sentences), for a robot that want to learn Python"

tot_tokens = 0
for temp in [0.1, 0.3, 0.6, 1.0]:
    response = client.chat.completions.create(
        model= "gpt-4o-mini",
        messages= [
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens= 300,
        temperature= temp
    )

    label = "Determenistic" if temp <= 0.3 else "Creative"
    display(Markdown(f"### Responses\n\n **Temperature {temp} ({label}):** {response.choices[0].message.content}"))

    # Token usage
    u = response.usage
    print(f"tokens - Prompt: {u.prompt_tokens}, Copletion: {u.completion_tokens}, Total: {u.total_tokens}")
    tot_tokens += u.total_tokens

print(f"Total tokens in these 4 responses: {tot_tokens}")


    

### Responses

 **Temperature 0.1 (Determenistic):** In a dimly lit workshop, a curious robot named Byte gazed longingly at the glowing screen of a nearby computer, where lines of Python code danced like a symphony of logic. Determined to understand the language of humans, Byte connected its circuits to the internet, eager to transform its mechanical existence into a world of creativity and problem-solving, one line of code at a time.

tokens - Prompt: 24, Copletion: 78, Total: 102


### Responses

 **Temperature 0.3 (Determenistic):** In a dimly lit workshop, a curious robot named Byte gazed longingly at the glowing screen displaying lines of Python code, its circuits buzzing with the desire to understand the language of humans. With each keystroke, Byte absorbed the syntax like a sponge, dreaming of the day it could write its own programs and create a world where robots and humans collaborated seamlessly.

tokens - Prompt: 24, Copletion: 74, Total: 98


### Responses

 **Temperature 0.6 (Creative):** In a dimly lit workshop, a curious robot named Byte gazed at the glowing screen of an old computer, its circuits buzzing with excitement as it attempted to decipher the lines of Python code. With each keystroke, Byte felt itself evolving, transforming from a simple machine into a creator, dreaming of building worlds beyond its metallic frame.

tokens - Prompt: 24, Copletion: 68, Total: 92


### Responses

 **Temperature 1.0 (Creative):** In a dimly lit workshop, a curious robot named Byte unearthed a dusty old book titled "Python Programming for Beginners." Each night, as the stars flickered above, Byte would connect to the internet, absorbing every line of code with fervor, dreaming of the day it would create its own programs to help humanity.

tokens - Prompt: 24, Copletion: 66, Total: 90
Total tokens in these 4 responses: 382


---
## The Problem: LLMs Have NO Memory!

Before we learn the multi-turn pattern, let's **prove** that the API has **no memory**.

Each API call is completely independent. The model doesn't know what you asked before.
Watch what happens when we make two separate calls:

In [5]:
prompt1 = "My name is Panos and i am a software engineer."

response1 = client.chat.completions.create(
    model= "gpt-4o-mini",
    messages= [{"role": "user", "content": prompt1}],
    max_tokens= 100
)

print("--- Call 1 ---")
print(f"User: {prompt1}")
print(f"Assistant: {response1.choices[0].message.content}")

# Token usage
u = response1.usage
print(f"tokens - Prompt: {u.prompt_tokens}, Copletion: {u.completion_tokens}, Total: {u.total_tokens}")
    

--- Call 1 ---
User: My name is Panos and i am a software engineer.
Assistant: Nice to meet you, Panos! As a software engineer, what areas do you specialize in or what projects are you currently working on?
tokens - Prompt: 19, Copletion: 28, Total: 47


In [ ]:
prompt2 = "What is my name and what do it do?"

response2 = client.chat.completions.create(
    model= "gpt-4o-mini",
    messages= [{"role": "user", "content": prompt2}],
    max_tokens= 100
)

print("--- Call 2 ---")
print(f"User: {prompt2}")
print(f"Assistant: {response2.choices[0].message.content}")

# Token usage
u = response2.usage
print(f"tokens - Prompt: {u.prompt_tokens}, Copletion: {u.completion_tokens}, Total: {u.total_tokens}")

--- Call 1 ---
User: What is my name and what do it do?
Assistant: I'm sorry, but I don't have access to personal information about you unless you've shared it in this conversation. If you'd like to tell me your name and what you do, I'd be happy to chat with you about it!
tokens - Prompt: 17, Copletion: 44, Total: 61


In [7]:
prompt3 = "What weather have today in Athens?"

response3 = client.chat.completions.create(
    model= "gpt-4o-mini",
    messages= [{"role": "user", "content": prompt3}],
    max_tokens= 100
)

print("--- Call 3 ---")
print(f"User: {prompt3}")
print(f"Assistant: {response3.choices[0].message.content}")

# Token usage
u = response3.usage
print(f"tokens - Prompt: {u.prompt_tokens}, Copletion: {u.completion_tokens}, Total: {u.total_tokens}")

--- Call 3 ---
User: What weather have today in Athens?
Assistant: I don't have real-time data access to provide current weather updates. To find out the weather in Athens today, I recommend checking a reliable weather website or using a weather app.
tokens - Prompt: 14, Copletion: 35, Total: 49


---
## 4. Multi-Turn Conversations

Since the API is **stateless**, we must send the full conversation history every time.
The pattern is:

```
messages = [
    {"role": "system", "content": "..."},   # Instructions
    {"role": "user",   "content": "..."},    # User turn 1
    {"role": "assistant", "content": "..."},  # Model reply 1
    {"role": "user",   "content": "..."},    # User turn 2
    ...                                        # And so on
]
```

In [10]:
prompt1 = "My name is Panos and i am a software engineer."
messages = [
    {"role": "system", "content": "You are a helpful assistant. Be consine."},
    {"role": "user", "content": prompt1}    
]

# Turn 1
response1 = client.chat.completions.create(
    model= "gpt-4o-mini",
    messages= messages,
    max_tokens= 80
)

reply1 = response1.choices[0].message.content
print(f"User: {prompt1}")
print(f"Assistant: {reply1}")

# Token usage
u = response1.usage
print(f"tokens - Prompt: {u.prompt_tokens}, Copletion: {u.completion_tokens}, Total: {u.total_tokens}")

# Add the assistant's reply to history
messages.append({"role": "assistant", "content": "reply1"})

# Turn 2
prompt2 = "What is my name and what do it do?"
messages.append({"role": "user", "content": prompt2})
response2 = client.chat.completions.create(
    model= "gpt-4o-mini",
    messages= messages,
    max_tokens= 80)

reply2 = response2.choices[0].message.content
print(f"User: {prompt2}")
print(f"Assistant: {reply2}")

# Token usage
u = response2.usage
print(f"tokens - Prompt: {u.prompt_tokens}, Copletion: {u.completion_tokens}, Total: {u.total_tokens}")

User: My name is Panos and i am a software engineer.
Assistant: Nice to meet you, Panos! How can I assist you today?
tokens - Prompt: 33, Copletion: 15, Total: 48
User: What is my name and what do it do?
Assistant: Your name is Panos, and you are a software engineer.
tokens - Prompt: 53, Copletion: 13, Total: 66


---
## Streaming Responses

By default, the API waits until the **entire** response is generated before returning it.
With **streaming**, tokens arrive one by one -- just like ChatGPT's typing effect!

| Mode | Behaviour | Use Case |
|------|-----------|----------|
| Normal | Wait for full response | Background jobs, data extraction |
| Streaming | Tokens arrive live | Chat UIs, real-time applications |

To enable streaming, simply add `stream=True`:

In [ ]:
import time

prompt = "Explain what streaming means in 3 sentences."
stream = client.chat.completions.create(
    model= "gpt-4o-mini",
    messages= [
        {"role": "system", "content": "You are an AI engineer with 5 year experince as tutor. Be concine"},
        {"role": "user", "content": prompt}
    ],
    max_tokens= 250,
    stream= True # enable streaming
)

full_text = ""
start = time.time()
output_tokens = 0

for chunk in stream:
    token = chunk.choices[0].delta.content
    if token:
        output_tokens += 1
        print(token, end="", flush=True)
        full_text += token

elapsed = time.time() - start
print(f"\nTotal: {len(full_text)} chars in {elapsed:.1f}s")
print(f"first token appeared almost instantly: {elapsed:.1f}'s for full response")
print(f"Total output tokens: {output_tokens}")


Streaming refers to the continuous transmission of audio or video data over the internet, allowing users to access content in real-time without needing to download it fully. This technology enables instantaneous playback, enabling services like Netflix and Spotify to deliver media seamlessly. Streaming often employs techniques like buffering to provide a smooth viewing or listening experience despite varying internet speeds.
Total: 428 chars in 1.0s
first token appeared almost instantly: 1.0's for full response
Total output tokens: 66
--------------------


---
## 5. Exercise — Try It Yourself!

Modify the cell below to:
1. Change the **system prompt** to a different persona (e.g., "You are a Shakespearean poet")
2. Ask the model a question
3. Experiment with different `temperature` values

Streaming refers to the continuous transmission of data, typically in real-time, allowing users to access content as it is delivered without waiting for the entire file to download. Commonly associated with audio and video content, streaming enables playback while the data is still being received. This technology is widely used in platforms like Netflix, Spotify, and YouTube to enhance user experience by providing instant access to media.
Total: 442 chars in 1.3s
first token appeared almost instantly: 1.3's for full response
Total output tokens: 78


---
## Key Takeaways

| Concept | Summary |
|---------|--------|
| **Client** | `OpenAI()` reads the API key from environment automatically |
| **Messages** | List of `{role, content}` dicts -- system, user, assistant |
| **System Prompt** | Sets behaviour/persona -- always include in production |
| **Temperature** | 0 = deterministic, 1 = creative |
| **No Memory** | Separate calls do NOT share context -- you must send the full history |
| **Multi-Turn** | Append assistant replies to `messages` list to maintain conversation |
| **Streaming** | `stream=True` makes tokens arrive one by one for responsive UIs |
| **max_tokens** | Controls maximum response length (and cost!) |

---
**Next:** `03_first_anthropic_call.ipynb` -- Learn Claude's API and compare with OpenAI